# Proprioceptive sensing — object stiffness estimation

The hand squeezes a grasped object with the four fingers held at constant stiffness
$k_{\rm hold}$ = {K_TIP_HOLD} N/m (clamping side).  The thumb alone sweeps through
increasing tip stiffness levels and the resulting position–force finite difference
estimates object compliance without external force sensors.

---

## Why $\Delta x / \Delta F$ recovers $C_O$

At equilibrium, the VMC spring force equals the object reaction force.  Taking the
finite difference between the gentle baseline ($k_1$) and each probe level ($k_2 > k_1$):

$$\Delta x_\text{thumb} = x(k_2) - x(k_1), \qquad \Delta F_\text{thumb} = F(k_2) - F(k_1)$$

$$C_O = \frac{\|\Delta x_\text{thumb}\|}{\|\Delta F_\text{thumb}\|}$$

The hand compliance cancels exactly; lower $C_O$ means stiffer object.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.join('../..'))
sys.path.insert(0, '.')
from hand_config import K_TIP_GENTLE, K_TIP_SWEEP, K_TIP_HOLD, OBJECTS, FINGERTIPS

OUTPUT_DIR = os.path.join('outputs', 'object_stiffness_hand')

K_TIP_GENTLE = float(K_TIP_GENTLE)
K_TIP_SWEEP  = [float(k) for k in K_TIP_SWEEP]
FINGERS = list(FINGERTIPS)

OBJECT_LABELS = ['Sphere', 'Sponge', 'Yarn']
OBJ_COLORS    = {'hard': '#D55E00', 'medium': '#E69F00', 'soft': '#56B4E9'}

def latest_run(object_name):
    path = os.path.join(OUTPUT_DIR, f'object_stiffness_hand_{object_name}.csv')
    return path if os.path.exists(path) else None

In [ ]:
def steady_state(df):
    out = {}
    for k_tip, sub in df.groupby('k_tip_Npm'):
        pos  = sub[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
        ref  = sub[['ref_thumb_x_m', 'ref_thumb_y_m', 'ref_thumb_z_m']].median().to_numpy()
        F1   = sub[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()
        out[float(k_tip)] = dict(pos=pos, ref=ref, F1=F1)
    return out


def thumb_compliance_dist(df):
    """Per-row C_O distribution for the thumb at each sweep level.

    Baseline (pos_b, F_b) is the median of the gentle phase.
    Returns dict: k_tip -> array of C_O values [m/N] (one per logged row).
    """
    gentle = df[df['k_tip_Npm'] == K_TIP_GENTLE]
    pos_b  = gentle[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
    F_b    = gentle[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()

    dist = {}
    for k in K_TIP_SWEEP:
        sub  = df[df['k_tip_Npm'] == k]
        rows = []
        for _, row in sub.iterrows():
            pos_i = np.array([row['tip_thumb_x_m'], row['tip_thumb_y_m'], row['tip_thumb_z_m']])
            F_i   = np.array([row['force_1st_thumb_x_N'], row['force_1st_thumb_y_N'], row['force_1st_thumb_z_N']])
            d_pos = pos_i - pos_b
            d_F   = F_i   - F_b
            nF    = np.linalg.norm(d_F)
            rows.append(np.linalg.norm(d_pos) / nF if nF > 1e-12 else np.nan)
        dist[float(k)] = np.array(rows)
    return dist


# Load latest run per object
data = {}
print('Loading data:')
for obj in OBJECTS:
    p = os.path.join(OUTPUT_DIR, f'object_stiffness_hand_{obj}.csv')
    if not os.path.exists(p):
        print(f'  {obj:<10s}: no data')
        continue
    df = pd.read_csv(p)
    ss   = steady_state(df)
    dist = thumb_compliance_dist(df)
    data[obj] = dict(df=df, ss=ss, dist=dist)
    print(f'  {obj:<10s}: k_tip levels {sorted(ss.keys())}')

## Thumb object compliance $C_O$ vs $k_{tip}$

$C_O^{\rm thumb}(k) = \|\Delta x_{\rm thumb}\| / \|\Delta F_{\rm thumb}\|$ for each sweep level.
Lower = stiffer object.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for obj in OBJECTS:
    if obj not in data:
        continue
    dist = data[obj]['dist']
    k_levels = sorted(dist.keys())
    medians = np.array([np.nanmedian(dist[k]) for k in k_levels]) * 1e3
    lo      = np.array([np.nanpercentile(dist[k], 25) for k in k_levels]) * 1e3
    hi      = np.array([np.nanpercentile(dist[k], 75) for k in k_levels]) * 1e3
    oi   = OBJECTS.index(obj)
    name = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
    color = OBJ_COLORS[obj]
    ax.fill_between(k_levels, lo, hi, alpha=0.20, color=color)
    ax.plot(k_levels, medians, marker='o', label=name, color=color, lw=2)

ax.set_xlabel(r'$k_{tip}$ [N/m]')
ax.set_ylabel(r'$C_O^{\mathrm{thumb}}$ [mm/N]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_compliance_sweep.pdf'), bbox_inches='tight')
plt.show()

## Thumb tip displacement $\|\Delta x_{\rm thumb}\|$ vs $k_{tip}$

Displacement of the thumb tip relative to the gentle baseline position.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for obj in OBJECTS:
    if obj not in data:
        continue
    df   = data[obj]['df']
    gentle = df[df['k_tip_Npm'] == K_TIP_GENTLE]
    pos_b  = gentle[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()

    k_levels, medians, lo, hi = [], [], [], []
    for k in K_TIP_SWEEP:
        sub  = df[df['k_tip_Npm'] == k]
        vals = sub[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].apply(
            lambda row: np.linalg.norm(row.to_numpy() - pos_b) * 1e3, axis=1)
        k_levels.append(k)
        medians.append(vals.median())
        lo.append(vals.quantile(0.25))
        hi.append(vals.quantile(0.75))

    color = OBJ_COLORS[obj]
    oi    = OBJECTS.index(obj)
    name  = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
    ax.fill_between(k_levels, lo, hi, alpha=0.20, color=color)
    ax.plot(k_levels, medians, marker='o', label=name, color=color, lw=2)

ax.set_xlabel(r'$k_{tip}$ [N/m]')
ax.set_ylabel(r'$\|\Delta x_{\mathrm{thumb}}\|$ [mm]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_displacement_sweep.pdf'), bbox_inches='tight')
plt.show()

## Thumb contact force $\|\Delta F_{\rm thumb}\|$ vs $k_{tip}$

Force change relative to the gentle baseline. Larger $\|\Delta F\|$ means firmer contact and a more reliable $C_O$ estimate.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for obj in OBJECTS:
    if obj not in data:
        continue
    df   = data[obj]['df']
    gentle = df[df['k_tip_Npm'] == K_TIP_GENTLE]
    F_b    = gentle[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()

    k_levels, medians, lo, hi = [], [], [], []
    for k in K_TIP_SWEEP:
        sub  = df[df['k_tip_Npm'] == k]
        vals = sub[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].apply(
            lambda row: np.linalg.norm(row.to_numpy() - F_b), axis=1)
        k_levels.append(k)
        medians.append(vals.median())
        lo.append(vals.quantile(0.25))
        hi.append(vals.quantile(0.75))

    color = OBJ_COLORS[obj]
    oi    = OBJECTS.index(obj)
    name  = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
    ax.fill_between(k_levels, lo, hi, alpha=0.20, color=color)
    ax.plot(k_levels, medians, marker='o', label=name, color=color, lw=2)

ax.set_xlabel(r'$k_{tip}$ [N/m]')
ax.set_ylabel(r'$\|\Delta F_{\mathrm{thumb}}\|$ [N]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_force_sweep.pdf'), bbox_inches='tight')
plt.show()

## Object classification by thumb compliance

$C_O^{\rm thumb}$ at the highest $k_{tip}$ level ranks the objects: lower = stiffer.

In [ ]:
K_TOP = max(K_TIP_SWEEP)

C_O_thumb = {}
for obj in OBJECTS:
    if obj not in data:
        continue
    dist = data[obj]['dist']
    if K_TOP not in dist:
        continue
    C_O_thumb[obj] = float(np.nanmedian(dist[K_TOP]))

if not C_O_thumb:
    print('No data available -- skipping object classification.')
else:
    sorted_objs    = sorted(C_O_thumb, key=C_O_thumb.get)
    cluster_labels = {sorted_objs[i]: lbl
                      for i, lbl in enumerate(['stiff', 'medium', 'soft'][:len(sorted_objs)])}

    vals       = [C_O_thumb[o] for o in sorted_objs]
    thresholds = [(vals[i] + vals[i+1]) / 2 for i in range(len(vals) - 1)]

    fig, ax = plt.subplots(figsize=(7, 3))
    for obj in sorted_objs:
        oi   = OBJECTS.index(obj)
        name = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
        lbl  = cluster_labels[obj]
        # IQR as error bar
        lo   = np.nanpercentile(data[obj]['dist'][K_TOP], 25) * 1e3
        hi   = np.nanpercentile(data[obj]['dist'][K_TOP], 75) * 1e3
        med  = C_O_thumb[obj] * 1e3
        ax.errorbar([med], [0], xerr=[[med - lo], [hi - med]],
                    fmt='o', color=OBJ_COLORS[obj], ms=10, capsize=6, lw=2)
        ax.annotate(f'{name}\n({lbl})', (med, 0),
                    textcoords='offset points', xytext=(0, 22),
                    ha='center', fontsize=11)
    for thr in thresholds:
        ax.axvline(thr * 1e3, color='0.4', lw=1.2, linestyle='--')

    ax.set_xlabel(r'$C_O^{\mathrm{thumb}}$ at $k_{tip}=' + f'{K_TOP:.0f}$' + r' N/m  [mm/N]  (median ± IQR)')
    ax.set_yticks([])
    ax.spines[['left', 'top', 'right']].set_visible(False)
    fig.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, 'object_classification.pdf'), bbox_inches='tight')
    plt.show()

    print('Object classification (thumb C_O, median):')
    for obj in sorted_objs:
        oi   = OBJECTS.index(obj)
        name = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
        vals_k = data[obj]['dist'][K_TOP] * 1e3
        print(f'  {name:<10s}  median={np.nanmedian(vals_k):.2f}  IQR=[{np.nanpercentile(vals_k,25):.2f}, {np.nanpercentile(vals_k,75):.2f}] mm/N  --> {cluster_labels[obj]}')